In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SESSION 1: GBM — CVaR v2.5 Curriculum Learning                            ║
# ║ 60 runs: 2 options × 3κ × 10 seeds                                        ║
# ║ Each run: MSE 500ep → CVaR fine-tune 200ep                                ║
# ║ Saves both MSE and CVaR metrics per run                                    ║
# ║ Expected: ~10h on T4                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import numpy as np, torch, torch.nn as nn, torch.optim as optim
import time, gc, os, json, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True; torch.cuda.empty_cache()

def clear_mem():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_FOLDER = '/content/drive/MyDrive/PHD_SBTS'
SESSION_NAME = 'GBM'
RESULTS_FILE = os.path.join(DRIVE_FOLDER, f'results_v25_{SESSION_NAME}.json')

# ═══════════════════════════════════════════════════════════════
#  LOAD DATA — GBM + Historical
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(f"LOADING DATA — {SESSION_NAME}")
print("=" * 70)

_d = np.load(os.path.join(DRIVE_FOLDER, 'gbm_deep_hedging.npz'))
S_train = torch.tensor(_d['S_train'], dtype=torch.float32, device=DEVICE)
S_val   = torch.tensor(_d['S_val'],   dtype=torch.float32, device=DEVICE)
S_test  = torch.tensor(_d['S_test'],  dtype=torch.float32, device=DEVICE)
print(f"  {SESSION_NAME}: train {_d['S_train'].shape}")
del _d

_, T_plus_1, d = S_train.shape
T = T_plus_1 - 1; N_ASSETS = d

hist_test = np.load(os.path.join(DRIVE_FOLDER, 'historical_test_paths.npz'))
hist_periods = {}
for pname in hist_test['period_names']:
    ps = str(pname)
    hist_periods[ps] = torch.tensor(hist_test[f'S_norm_{ps}'],
                                     dtype=torch.float32, device=DEVICE)
    print(f"  Hist {ps}: {hist_periods[ps].shape}")

# ═══════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════
COST_RATE = 0.001; BATCH_SIZE = 4096; CVAR_ALPHA = 0.95
MSE_EPOCHS = 500; MSE_LR = 1e-3; MSE_PATIENCE = 20
CVAR_EPOCHS = 200; CVAR_LR = 1e-4; CVAR_PATIENCE = 20

OPTION_NAMES = ['basket_asian_call', 'asian_worst_of_put']
KAPPA_LEVELS = [0.95, 1.00, 1.05]
N_SEEDS = 10

total_runs = len(OPTION_NAMES) * len(KAPPA_LEVELS) * N_SEEDS
print(f"\n  Session: {SESSION_NAME} — {total_runs} curriculum runs")
print(f"  MSE {MSE_EPOCHS}ep → CVaR {CVAR_EPOCHS}ep fine-tune")

# ═══════════════════════════════════════════════════════════════
#  COMPONENTS
# ═══════════════════════════════════════════════════════════════
def compute_running_averages(S_paths):
    M, T1, d = S_paths.shape; T = T1 - 1
    avg = torch.zeros(M, T, d, device=S_paths.device)
    avg[:, 0, :] = S_paths[:, 0, :]
    if T > 1:
        cumsum = S_paths[:, 1:, :].cumsum(dim=1)
        counts = torch.arange(1, T+1, device=S_paths.device,
                              dtype=torch.float32).unsqueeze(0).unsqueeze(2)
        avg[:, 1:, :] = (cumsum / counts)[:, :-1, :]
    return avg

def payoff_basket_asian_call(S_paths, kappa=1.0):
    R_bar = S_paths[:, 1:, :].mean(dim=1)
    return torch.clamp(R_bar.mean(dim=1) - kappa, min=0.0)

def payoff_asian_worst_of_put(S_paths, kappa=1.0):
    R_bar = S_paths[:, 1:, :].mean(dim=1)
    return torch.clamp(kappa - R_bar.min(dim=1).values, min=0.0)

PAYOFF_FNS = {
    'basket_asian_call': payoff_basket_asian_call,
    'asian_worst_of_put': payoff_asian_worst_of_put,
}

class HedgingNetwork(nn.Module):
    def __init__(self, d=3, hidden=(64, 64)):
        super().__init__()
        self.d = d; layers = []; prev = 3*d+1
        for h in hidden: layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        layers.append(nn.Linear(prev, d))
        self.net = nn.Sequential(*layers)
        nn.init.xavier_uniform_(self.net[-1].weight, gain=0.1)
        nn.init.zeros_(self.net[-1].bias)
        self.V0 = nn.Parameter(torch.tensor(0.0))
    def forward(self, spots, running_avg, delta_prev, time_left):
        return self.net(torch.cat([spots, running_avg, delta_prev, time_left], dim=1))

def loss_mse(residuals): return (residuals ** 2).mean()

class CVaRLoss(nn.Module):
    def __init__(self, alpha=0.95):
        super().__init__(); self.alpha = alpha
        self.nu = nn.Parameter(torch.tensor(0.0))
    def forward(self, residuals):
        excess = torch.clamp(residuals - self.nu, min=0.0)
        return self.nu + excess.mean() / (1.0 - self.alpha)

def deep_hedge_forward(net, S_paths, payoff_fn, kappa=1.0, cost_rate=0.001):
    M, T1, d = S_paths.shape; T = T1 - 1
    running_avg = compute_running_averages(S_paths)
    time_fracs = torch.arange(T, 0, -1, device=S_paths.device, dtype=torch.float32) / T
    delta = torch.zeros(M, d, device=S_paths.device)
    pnl = torch.zeros(M, device=S_paths.device)
    cost = torch.zeros(M, device=S_paths.device)
    for t in range(T):
        delta_new = net(S_paths[:,t,:], running_avg[:,t,:], delta, time_fracs[t].expand(M,1))
        cost = cost + cost_rate * ((delta_new - delta).abs() * S_paths[:,t,:]).sum(dim=1)
        pnl = pnl + (delta_new * (S_paths[:,t+1,:] - S_paths[:,t,:])).sum(dim=1)
        delta = delta_new
    payoff = payoff_fn(S_paths, kappa)
    return {'residuals': payoff - net.V0 - pnl + cost, 'payoff': payoff.detach(),
            'pnl': pnl.detach(), 'cost': cost.detach(), 'V0': net.V0.item()}

@torch.no_grad()
def compute_metrics(net, S_paths, payoff_fn, kappa=1.0, cost_rate=0.001):
    net.eval(); r = deep_hedge_forward(net, S_paths, payoff_fn, kappa, cost_rate)
    res = r['residuals']; n = len(res)
    s = torch.sort(res).values
    return {'rmse': (res**2).mean().sqrt().item(), 'mean': res.mean().item(),
            'std': res.std().item(),
            'cvar95': s[int(np.ceil(0.95*n)):].mean().item(),
            'cvar99': s[int(np.ceil(0.99*n)):].mean().item(),
            'max': res.max().item(), 'V0': r['V0'],
            'avg_cost': r['cost'].mean().item()}

# ═══════════════════════════════════════════════════════════════
#  TRAINING LOOP
# ═══════════════════════════════════════════════════════════════
M_train = S_train.shape[0]
n_batches = max(1, M_train // BATCH_SIZE)

def train_loop(net, loss_fn, all_params, payoff_fn, kappa, lr, max_ep, patience, label=""):
    optimizer = optim.Adam(all_params, lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5, min_lr=1e-7)
    best_val = float('inf'); pat_cnt = 0; best_state = None; t0 = time.time()
    for epoch in range(max_ep):
        net.train(); perm = torch.randperm(M_train, device=DEVICE); ep_loss = 0.0
        for b in range(n_batches):
            idx = perm[b*BATCH_SIZE:(b+1)*BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(deep_hedge_forward(net, S_train[idx], payoff_fn, kappa, COST_RATE)['residuals'])
            loss.backward(); torch.nn.utils.clip_grad_norm_(all_params, 1.0); optimizer.step()
            ep_loss += loss.item()
        ep_loss /= n_batches
        net.eval()
        with torch.no_grad():
            val_loss = loss_fn(deep_hedge_forward(net, S_val, payoff_fn, kappa, COST_RATE)['residuals']).item()
        scheduler.step(val_loss)
        if val_loss < best_val - 1e-6:
            best_val = val_loss; pat_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
        else: pat_cnt += 1
        if pat_cnt >= patience: break
    if best_state: net.load_state_dict(best_state)
    net.eval()
    return best_state, epoch + 1, time.time() - t0

# ═══════════════════════════════════════════════════════════════
#  CURRICULUM: MSE → CVaR for each config
# ═══════════════════════════════════════════════════════════════
def load_results():
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, 'r') as f: return json.load(f)
    return {}

def save_results(results):
    tmp = RESULTS_FILE + '.tmp'
    with open(tmp, 'w') as f: json.dump(results, f, indent=2)
    os.replace(tmp, RESULTS_FILE)

def make_key(opt, kappa, seed):
    return f"{SESSION_NAME}_v25_{opt}_k{kappa:.2f}_s{seed}"

all_results = load_results()
print(f"\n  Existing: {len(all_results)} / {total_runs}")

run_queue = []
for opt in OPTION_NAMES:
    for kappa in KAPPA_LEVELS:
        for seed in range(N_SEEDS):
            key = make_key(opt, kappa, seed)
            if key not in all_results:
                run_queue.append((opt, kappa, seed, key))

print(f"  Remaining: {len(run_queue)}")
if not run_queue:
    print("\n  ✅ All runs complete!")
else:
    t_start = time.time()
    for i, (opt, kappa, seed, key) in enumerate(run_queue):
        print(f"\n{'─' * 60}")
        print(f"  [{len(all_results)+1}/{total_runs}] {key}")

        torch.manual_seed(seed); np.random.seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed(seed)

        payoff_fn = PAYOFF_FNS[opt]
        net = HedgingNetwork(d=N_ASSETS, hidden=(64, 64)).to(DEVICE)

        # Phase 1: MSE
        mse_state, mse_ep, mse_time = train_loop(
            net, loss_mse, list(net.parameters()), payoff_fn, kappa,
            MSE_LR, MSE_EPOCHS, MSE_PATIENCE)
        test_mse = compute_metrics(net, S_test, payoff_fn, kappa, COST_RATE)
        hist_mse = {p: compute_metrics(net, S, payoff_fn, kappa, COST_RATE)
                    for p, S in hist_periods.items()}
        v0_mse = net.V0.item()
        print(f"    MSE: {mse_time:.0f}s ({mse_ep}ep) V₀={v0_mse:.4f} Std={test_mse['std']:.4f}")

        # Phase 2: CVaR fine-tune
        net.load_state_dict(mse_state)
        net.V0.requires_grad = False
        cvar_loss = CVaRLoss(alpha=CVAR_ALPHA).to(DEVICE)
        cvar_params = [p for p in net.parameters() if p.requires_grad] + list(cvar_loss.parameters())
        _, cvar_ep, cvar_time = train_loop(
            net, cvar_loss, cvar_params, payoff_fn, kappa,
            CVAR_LR, CVAR_EPOCHS, CVAR_PATIENCE)
        test_v25 = compute_metrics(net, S_test, payoff_fn, kappa, COST_RATE)
        hist_v25 = {p: compute_metrics(net, S, payoff_fn, kappa, COST_RATE)
                    for p, S in hist_periods.items()}
        print(f"    CVaR: {cvar_time:.0f}s ({cvar_ep}ep) ν={cvar_loss.nu.item():.4f} "
              f"Std={test_v25['std']:.4f} CVaR95={test_v25['cvar95']:.4f}")

        all_results[key] = {
            'key': key, 'ds': SESSION_NAME, 'option': opt, 'kappa': kappa, 'seed': seed,
            'mse': {'n_epochs': mse_ep, 'train_time': round(mse_time,1),
                    'test': test_mse, 'historical': hist_mse, 'V0': v0_mse},
            'cvar': {'n_epochs': cvar_ep, 'train_time': round(cvar_time,1),
                     'test': test_v25, 'historical': hist_v25,
                     'V0': net.V0.item(), 'nu': cvar_loss.nu.item()},
        }
        save_results(all_results)

        elapsed = time.time() - t_start
        remaining = elapsed / (i+1) * (len(run_queue) - i - 1)
        print(f"    💾 ({i+1}/{len(run_queue)}, ~{remaining/3600:.1f}h left)")

        del net, cvar_loss; clear_mem()

    print(f"\n\n{'═' * 70}")
    print(f"  ✅ {SESSION_NAME} SESSION COMPLETE: {len(all_results)} runs")
    print(f"  Results: {RESULTS_FILE}")
    print(f"{'═' * 70}")

Device: cuda
  GPU: Tesla T4
Mounted at /content/drive

LOADING DATA — GBM
  GBM: train (16000, 253, 3)
  Hist COVID_2020: torch.Size([253, 253, 3])
  Hist PostCOVID_2021_22: torch.Size([503, 253, 3])
  Hist Normal_2023_24: torch.Size([500, 253, 3])

  Session: GBM — 60 curriculum runs
  MSE 500ep → CVaR 200ep fine-tune

  Existing: 21 / 60
  Remaining: 39

────────────────────────────────────────────────────────────
  [22/60] GBM_v25_basket_asian_call_k1.05_s1
    MSE: 435s (471ep) V₀=0.0388 Std=0.0063
    CVaR: 185s (200ep) ν=0.0074 Std=0.0071 CVaR95=0.0106
    💾 (1/39, ~6.6h left)

────────────────────────────────────────────────────────────
  [23/60] GBM_v25_basket_asian_call_k1.05_s2
    MSE: 281s (310ep) V₀=0.0389 Std=0.0077
    CVaR: 182s (200ep) ν=0.0091 Std=0.0089 CVaR95=0.0135
    💾 (2/39, ~5.6h left)

────────────────────────────────────────────────────────────
  [24/60] GBM_v25_basket_asian_call_k1.05_s3
    MSE: 347s (383ep) V₀=0.0383 Std=0.0068
    CVaR: 182s (200ep) ν=0.